# 04 - SQL analysis in SQLite

**Goal:** load the clean data into a small SQL database and answer your project questions with SQL queries. Analyst interviews test SQL heavily, so this notebook doubles as practice.

**What is SQLite?** A database stored in one file, with nothing to install. Python talks to it through the built-in `sqlite3` module.

**Two tables:**
- `jobs`: one row per job (role, city, experience level, salary...)
- `job_skills`: one row per job-skill pair, linked to `jobs` by `job_id`

Run the cells **from top to bottom**. You need `jobs_final.csv` and `job_skills.csv` from the previous notebook.

| Query | Question | SQL idea you practise |
|---|---|---|
| 1 | Which cities hire the most? | `GROUP BY`, subquery |
| 2 | Which skills are most demanded per role? | `JOIN`, `COUNT(DISTINCT)` |
| 3 | What should a fresher learn? | CTE (`WITH`), filtering |
| 4 | Fresher vs experienced skills | `CASE WHEN` pivot |
| 5 | Which skills appear together? | self-join |
| 6 | What goes with Power BI? | CTE + join |
| 7 | Median salary by role | window functions |
| 8 | Median salary by experience level | window functions |
| 9 | Remote share by role | conditional aggregation |
| 10 | Power BI demand by city | join + group |

## Step 1 - Create the database

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

jobs = pd.read_csv("../data/clean/jobs_final.csv")
skills = pd.read_csv("../data/clean/job_skills.csv")

conn = sqlite3.connect("../data/clean/job_market.db")

jobs.to_sql("jobs", conn, if_exists="replace", index=False)
skills.to_sql("job_skills", conn, if_exists="replace", index=False)

conn.execute("CREATE INDEX IF NOT EXISTS idx_jobs_id ON jobs(job_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_skills_job ON job_skills(job_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_skills_skill ON job_skills(skill)")
conn.commit()

def run(sql):
    """Run a SQL query and return the result as a table."""
    return pd.read_sql_query(sql, conn)

ALL_QUERIES = []   # we collect every query here and save them to a .sql file at the end

print(run("SELECT 'jobs' AS table_name, COUNT(*) AS rows_count FROM jobs UNION ALL SELECT 'job_skills', COUNT(*) FROM job_skills"))

In [ ]:
run("SELECT role_group, COUNT(*) AS jobs FROM jobs GROUP BY role_group ORDER BY jobs DESC")

## Query 1 - Which cities have the most openings? (question 1)
Only core roles, and we leave out jobs that do not name a city. The subquery in the middle works out the total so we can show percentages.

In [ ]:
q = """
SELECT city,
       COUNT(*) AS jobs,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM jobs WHERE is_core_role = 1 AND city <> 'Not specified'), 1) AS pct_of_located_jobs
FROM jobs
WHERE is_core_role = 1
  AND city <> 'Not specified'
GROUP BY city
ORDER BY jobs DESC
LIMIT 10
"""
ALL_QUERIES.append(("Query 1 - Top cities for core data roles", q))
run(q)

In [ ]:
run("""
SELECT COUNT(*) AS core_jobs,
       SUM(city = 'Not specified') AS no_city,
       ROUND(100.0 * SUM(city = 'Not specified') / COUNT(*), 1) AS pct_no_city
FROM jobs WHERE is_core_role = 1
""")

## Query 2 - Most demanded skills by role (question 2)
`JOIN` connects each job to its skills. We count **distinct** jobs, and divide by the number of jobs in the role. This query shows the top skills for **Data Analyst** jobs; change the role name in the `WHERE` line to see the others.

In [ ]:
q = """
SELECT s.skill,
       s.category,
       COUNT(DISTINCT s.job_id) AS jobs,
       ROUND(100.0 * COUNT(DISTINCT s.job_id) / (SELECT COUNT(*) FROM jobs WHERE role_group = 'Data Analyst'), 1) AS pct_of_role_jobs
FROM job_skills s
JOIN jobs j ON j.job_id = s.job_id
WHERE j.role_group = 'Data Analyst'
GROUP BY s.skill, s.category
ORDER BY jobs DESC
LIMIT 15
"""
ALL_QUERIES.append(("Query 2 - Most demanded skills for Data Analyst jobs", q))
run(q)

## Query 3 - What should a fresher learn first? (question 5)
A **CTE** (`WITH ... AS`) gives a name to an intermediate result. Here `fresher` is the set of entry-level analyst-type jobs.
We drop soft skills because they are not something you "learn first".

In [ ]:
q = """
WITH fresher AS (
    SELECT job_id
    FROM jobs
    WHERE role_group IN ('Data Analyst', 'BI Analyst', 'Business Analyst')
      AND experience_level = 'Entry/Fresher'
)
SELECT s.skill,
       s.category,
       COUNT(*) AS jobs,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM fresher), 1) AS pct_of_fresher_jobs
FROM job_skills s
JOIN fresher f ON f.job_id = s.job_id
WHERE s.category <> 'Soft Skill'
GROUP BY s.skill, s.category
ORDER BY jobs DESC
LIMIT 12
"""
ALL_QUERIES.append(("Query 3 - Skills to learn first for fresher analyst jobs", q))
run(q)

## Query 4 - Fresher vs experienced: how do requirements change? (question 4)
`CASE WHEN` lets us put two groups side by side as two columns. The last column shows how many percentage points higher the demand is for mid/senior roles.

In [ ]:
q = """
WITH base AS (
    SELECT job_id,
           CASE WHEN experience_level = 'Entry/Fresher' THEN 'Fresher' ELSE 'Mid/Senior' END AS grp
    FROM jobs
    WHERE role_group IN ('Data Analyst', 'BI Analyst', 'Business Analyst')
      AND experience_level IN ('Entry/Fresher', 'Mid (2-4 yrs)', 'Senior (5+ yrs)')
),
totals AS (
    SELECT grp, COUNT(*) AS n FROM base GROUP BY grp
)
SELECT s.skill,
       ROUND(100.0 * SUM(CASE WHEN b.grp = 'Fresher' THEN 1 ELSE 0 END)
             / (SELECT n FROM totals WHERE grp = 'Fresher'), 1) AS fresher_pct,
       ROUND(100.0 * SUM(CASE WHEN b.grp = 'Mid/Senior' THEN 1 ELSE 0 END)
             / (SELECT n FROM totals WHERE grp = 'Mid/Senior'), 1) AS mid_senior_pct,
       ROUND(100.0 * SUM(CASE WHEN b.grp = 'Mid/Senior' THEN 1 ELSE 0 END)
             / (SELECT n FROM totals WHERE grp = 'Mid/Senior')
           - 100.0 * SUM(CASE WHEN b.grp = 'Fresher' THEN 1 ELSE 0 END)
             / (SELECT n FROM totals WHERE grp = 'Fresher'), 1) AS gap_points
FROM base b
JOIN job_skills s ON s.job_id = b.job_id
WHERE s.category <> 'Soft Skill'
GROUP BY s.skill
ORDER BY mid_senior_pct DESC
LIMIT 15
"""
ALL_QUERIES.append(("Query 4 - Fresher vs mid/senior skill demand (analyst-type roles)", q))
run(q)

In [ ]:
run("""
SELECT CASE WHEN experience_level = 'Entry/Fresher' THEN 'Fresher' ELSE 'Mid/Senior' END AS grp,
       COUNT(*) AS jobs
FROM jobs
WHERE role_group IN ('Data Analyst', 'BI Analyst', 'Business Analyst')
  AND experience_level IN ('Entry/Fresher', 'Mid (2-4 yrs)', 'Senior (5+ yrs)')
GROUP BY grp
""")

## Query 5 - Which skills appear together? (question 3)
A **self-join** joins `job_skills` to itself: for every job we pair each skill with every other skill in the same job.
`a.skill < b.skill` keeps each pair once (otherwise we would get both SQL+Python and Python+SQL).

In [ ]:
q = """
SELECT a.skill AS skill_1,
       b.skill AS skill_2,
       COUNT(*) AS jobs
FROM job_skills a
JOIN job_skills b ON a.job_id = b.job_id AND a.skill < b.skill
JOIN jobs j ON j.job_id = a.job_id
WHERE j.is_core_role = 1
GROUP BY a.skill, b.skill
ORDER BY jobs DESC
LIMIT 15
"""
ALL_QUERIES.append(("Query 5 - Most common skill pairs (core roles)", q))
run(q)

## Query 6 - What do employers ask for alongside Power BI?
First we find all jobs that mention Power BI, then count the other skills in those jobs. This tells you what to learn **with** Power BI.

In [ ]:
q = """
WITH pbi AS (
    SELECT s.job_id
    FROM job_skills s
    JOIN jobs j ON j.job_id = s.job_id
    WHERE s.skill = 'Power BI' AND j.is_core_role = 1
)
SELECT s.skill,
       COUNT(*) AS jobs_with_powerbi,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM pbi), 1) AS pct_of_powerbi_jobs
FROM job_skills s
JOIN pbi ON pbi.job_id = s.job_id
WHERE s.skill <> 'Power BI'
GROUP BY s.skill
ORDER BY jobs_with_powerbi DESC
LIMIT 12
"""
ALL_QUERIES.append(("Query 6 - Skills requested together with Power BI", q))
run(q)

## Query 7 - Median salary by role (window functions)
SQLite has no `MEDIAN` function, so we build one. `ROW_NUMBER()` numbers the salaries in order inside each role, `COUNT(*) OVER (...)` gives the group size, and we pick the middle row (or average the two middle rows).
This is a favourite interview topic. We use `salary_lpa_clean`, which has the suspicious salaries removed. LPA means lakhs per annum, and these are Glassdoor estimates.

In [ ]:
q = """
WITH ranked AS (
    SELECT role_group,
           salary_lpa_clean,
           ROW_NUMBER() OVER (PARTITION BY role_group ORDER BY salary_lpa_clean) AS rn,
           COUNT(*) OVER (PARTITION BY role_group) AS cnt
    FROM jobs
    WHERE is_core_role = 1 AND salary_lpa_clean IS NOT NULL
)
SELECT role_group,
       MAX(cnt) AS jobs_with_salary,
       ROUND(AVG(salary_lpa_clean), 2) AS median_lpa
FROM ranked
WHERE rn IN ((cnt + 1) / 2, (cnt + 2) / 2)
GROUP BY role_group
ORDER BY median_lpa DESC
"""
ALL_QUERIES.append(("Query 7 - Median salary (LPA) by role", q))
run(q)

## Query 8 - Median salary by experience level (question 4)
Same technique, grouped by experience level instead. Interns and trainees are left out so they do not drag down the Entry/Fresher figure.

In [ ]:
q = """
WITH ranked AS (
    SELECT experience_level,
           salary_lpa_clean,
           ROW_NUMBER() OVER (PARTITION BY experience_level ORDER BY salary_lpa_clean) AS rn,
           COUNT(*) OVER (PARTITION BY experience_level) AS cnt
    FROM jobs
    WHERE is_core_role = 1
      AND is_intern = 0
      AND salary_lpa_clean IS NOT NULL
      AND experience_level <> 'Not specified'
)
SELECT experience_level,
       MAX(cnt) AS jobs_with_salary,
       ROUND(AVG(salary_lpa_clean), 2) AS median_lpa
FROM ranked
WHERE rn IN ((cnt + 1) / 2, (cnt + 2) / 2)
GROUP BY experience_level
ORDER BY median_lpa
"""
ALL_QUERIES.append(("Query 8 - Median salary (LPA) by experience level, excluding interns", q))
run(q)

## Query 9 - How much of the market is remote?
`SUM(condition)` counts the rows where the condition is true, which is handy for percentages.

In [ ]:
q = """
SELECT role_group,
       COUNT(*) AS jobs,
       SUM(work_mode = 'Remote') AS remote_jobs,
       ROUND(100.0 * SUM(work_mode = 'Remote') / COUNT(*), 1) AS remote_pct
FROM jobs
WHERE is_core_role = 1
GROUP BY role_group
ORDER BY jobs DESC
"""
ALL_QUERIES.append(("Query 9 - Remote share by role", q))
run(q)

## Query 10 - Where is Power BI demand highest?
Cities with at least 10 core-role postings, ranked by the share that ask for Power BI.

In [ ]:
q = """
SELECT j.city,
       COUNT(*) AS core_jobs,
       SUM(CASE WHEN s.job_id IS NOT NULL THEN 1 ELSE 0 END) AS powerbi_jobs,
       ROUND(100.0 * SUM(CASE WHEN s.job_id IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS powerbi_pct
FROM jobs j
LEFT JOIN job_skills s ON s.job_id = j.job_id AND s.skill = 'Power BI'
WHERE j.is_core_role = 1 AND j.city <> 'Not specified'
GROUP BY j.city
HAVING COUNT(*) >= 10
ORDER BY powerbi_pct DESC
"""
ALL_QUERIES.append(("Query 10 - Power BI demand by city", q))
run(q)

## Step 2 - Save your queries to a .sql file
This writes all the queries above into `sql/analysis_queries.sql`, so your GitHub repo shows your SQL work.

In [ ]:
lines = ["-- India Data Job Market Intelligence: analysis queries (SQLite)", ""]
for title, query in ALL_QUERIES:
    lines.append("-- " + title)
    lines.append(query.strip() + ";")
    lines.append("")

Path("../sql/analysis_queries.sql").write_text("\n".join(lines), encoding="utf-8")
print("Saved", len(ALL_QUERIES), "queries to ../sql/analysis_queries.sql")

conn.close()

## Checkpoint
Send back the printed outputs of Queries 1, 3, 4, 7, 8 and 10 (plus the small check tables under Queries 1 and 4).